In [0]:
dbutils.widgets.text("catalog", "olist_dev")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = f"{CATALOG}.lakehouse"
print(f"schéma : {SCHEMA}")

In [0]:
%load_ext autoreload
%autoreload 2

import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from pyspark.sql import functions as F
from olist.bronze import ecrire_table
from olist.silver import problemes_de_cle
from olist.gold import dim_date, dim_geography, dim_customer, dim_seller, dim_product, fct_order_items, fct_orders

In [0]:
silver = lambda nom: spark.table(f"{SCHEMA}.silver_{nom}")
gold = lambda nom: spark.table(f"{SCHEMA}.{nom}")

ecrire_table(dim_date(spark), f"{SCHEMA}.dim_date")
ecrire_table(dim_geography(silver("geolocation")), f"{SCHEMA}.dim_geography")
ecrire_table(dim_customer(silver("customers"), silver("geolocation")), f"{SCHEMA}.dim_customer")
ecrire_table(dim_seller(silver("sellers"), silver("geolocation")), f"{SCHEMA}.dim_seller")
ecrire_table(dim_product(silver("products")), f"{SCHEMA}.dim_product")

ecrire_table(
    fct_order_items(silver("order_items"), silver("orders"), gold("dim_customer"), gold("dim_seller")),
    f"{SCHEMA}.fct_order_items",
)
ecrire_table(
    fct_orders(silver("orders"), gold("fct_order_items"), gold("dim_product"),
               silver("payments"), silver("reviews")),
    f"{SCHEMA}.fct_orders",
)
print("Gold écrit : 5 dimensions, 2 tables de faits")

In [0]:
attendu = {
    "dim_geography": (["zip_code_prefix"], 19_010),
    "dim_customer": (["customer_id"], 99_441),
    "dim_seller": (["seller_id"], 3_095),
    "dim_product": (["product_id"], 32_951),
    "fct_order_items": (["order_id", "order_item_id"], 112_650),
    "fct_orders": (["order_id"], 99_441),
}
erreurs = {}
for nom, (cle, lignes) in attendu.items():
    t = gold(nom)
    problemes = problemes_de_cle(t, cle)
    if t.count() != lignes:
        problemes.append(f"{t.count():,} lignes au lieu de {lignes:,}")
    print(f"{nom:<16} {t.count():>9,} lignes  {'OK' if not problemes else 'ERREUR : ' + ' ; '.join(problemes)}")
    if problemes:
        erreurs[nom] = problemes

# Intégrité référentielle : chaque clé étrangère existe dans sa dimension
liens = [
    ("fct_order_items", "product_id", "dim_product", "product_id"),
    ("fct_order_items", "seller_id", "dim_seller", "seller_id"),
    ("fct_orders", "customer_id", "dim_customer", "customer_id"),
    ("fct_orders", "date_key_achat", "dim_date", "date_key"),
]
for fait, cle_f, dim, cle_d in liens:
    cles_fait = gold(fait).select(F.col(cle_f).alias("cle"))
    cles_dim = gold(dim).select(F.col(cle_d).alias("cle"))
    orphelins = cles_fait.join(cles_dim, "cle", "left_anti").count()
    print(f"{fait}.{cle_f} → {dim} : {orphelins} orphelin(s)  {'OK' if orphelins == 0 else 'ERREUR'}")
    if orphelins:
        erreurs[f"{fait}.{cle_f}"] = orphelins

# Les montants ne doivent ni se perdre ni se dupliquer dans les agrégations
total_silver = silver("order_items").agg(F.sum("price")).first()[0]
total_gold = gold("fct_orders").agg(F.sum("montant_articles")).first()[0]
print(f"somme des prix : silver={total_silver:,.2f}  gold={total_gold:,.2f}  "
      f"{'OK' if total_silver == total_gold else 'ERREUR'}")
if total_silver != total_gold:
    erreurs["montants"] = (total_silver, total_gold)

assert not erreurs, f"Gold incorrect : {erreurs}"

In [0]:
f = gold("fct_orders")
display(f.agg(
    F.count("en_retard").alias("commandes_avec_cible"),
    F.sum("en_retard").alias("retards"),
    F.sum(F.col("distance_max_km").isNull().cast("int")).alias("sans_distance"),
))
display(
    f.filter(F.col("en_retard").isNotNull())
    .groupBy("vendeur_en_retard")
    .agg(F.count("*").alias("commandes"), F.round(F.avg("en_retard") * 100, 1).alias("taux_retard_pct"))
    .orderBy("vendeur_en_retard")
)